# Migration to GRIB2 - practical 2
## Parameter identification, units and wave streams

With the migration to GRIB2, ECMWF uses **WMO codes** wherever they exist. For most parameters, your MARS
request stays the same. This practical covers three groups that change:

| Case | What changes | Parameters |
| --- | --- | --- |
| 1 | identify parameters by **paramId**, not `indicatorOfParameter.table2Version` | 2 m temperature |
| 2 | **WMO units**: new paramId, same shortName | snow depth, total cloud cover, large-scale precipitation |
| 3 | wave parameters move from `stream=wave` to **`stream=oper`** | significant wave height, mean wave period |

All data come from **MARS**, for the same forecast, **1 September 2026 00 UTC**:  
- Mix of GRIB1 and GRIB2 (depending on the parameter) with `expver=1`, and
- the GRIB2 test data with **`expver=2250`**.

`expver` changes only because the GRIB2 data are a test dataset. Once GRIB2 is operational, it will be
`expver=1` again.

`grid` and `area` interpolate the data to 0.25° over Europe, to keep the downloads small.

## Setup

You need **earthkit-data** (with the mars source) and a recent **ecCodes under the hood**.
ecCodes older than 2.42.0 cannot decode the new encoding correctly - always use the latest release.

In [ ]:
import earthkit.data as ekd

# The keys we print for every field
keys = ["metadata.edition", "metadata.stream", "metadata.shortName", "metadata.paramId",
        "metadata.units", "metadata.name"]

## Identifying a parameter: use the paramId

A GRIB1 parameter is identified by two numbers: `indicatorOfParameter` and `table2Version`. That is why
MARS requests often use the notation `param=167.128`. In GRIB2 these two keys do not exist: a parameter is
identified by **discipline**, **parameterCategory** and **parameterNumber**, from the WMO code tables.
ecCodes translates both into the same ECMWF **paramId**.

### GRIB1

In [ ]:
t2m_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "1",                # changed
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "167.128",           # changed: indicatorOfParameter.table2Version notation
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
t2m_grib1.ls(keys=["metadata.edition", "metadata.shortName", "metadata.paramId",
                   "metadata.indicatorOfParameter", "metadata.table2Version"])

### GRIB2

In [ ]:
t2m_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "2250",             # changed (test data only)
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "167",               # changed: paramId
        "timespan": "none",           # NEW: instantaneous field
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
t2m_grib2.ls(keys=["metadata.edition", "metadata.shortName", "metadata.paramId",
                   "metadata.discipline", "metadata.parameterCategory", "metadata.parameterNumber",
                   "metadata.localTablesVersion"])

| | GRIB1 | GRIB2 | |
| --- | --- | --- | --- |
| `paramId` | 167 | 167 | same |
| identified in the data by | `indicatorOfParameter=167`, `table2Version=128` | `discipline=0`, `parameterCategory=0`, `parameterNumber=0` | changed |
| `localTablesVersion` | - | 0 (WMO code, no ECMWF local table needed) | new |

The `167.128` notation refers to keys that only exist in GRIB1. Always write the **paramId** in your requests
(`param=167`). It works for both editions.

## WMO units: new paramId, same shortName

Some parameters are encoded today in units that are not the WMO ones: metres of water equivalent instead of
kg m\*\*-2, fractions (0 - 1) instead of percent.   

In GRIB2 they are provided **only in WMO units**, with a
**new paramId**.

### GRIB1

In [ ]:
sd_tcc_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "1",                # changed
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "141/164",           # changed: sd in m of water equivalent, tcc in (0 - 1)
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
sd_tcc_grib1.ls(keys=keys)

### GRIB2

In [ ]:
sd_tcc_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "2250",             # changed (test data only)
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "228141/228164",     # changed: sd in kg m**-2, tcc in %
        "timespan": "none",           # NEW: instantaneous fields
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
sd_tcc_grib2.ls(keys=keys)

The **shortName is the same** (`sd`, `tcc`), but the paramId and the units are not. Compare the values:

In [ ]:
sd_tcc_grib1.ls(keys=["metadata.shortName", "metadata.paramId", "metadata.maximum", "metadata.units"])

In [ ]:
sd_tcc_grib2.ls(keys=["metadata.shortName", "metadata.paramId", "metadata.maximum", "metadata.units"])

| | GRIB1 | GRIB2 | |
| --- | --- | --- | --- |
| snow depth `param` | 141 | 228141 | changed |
| snow depth units | m of water equivalent | kg m\*\*-2 | changed: values are 1000 times larger |
| total cloud cover `param` | 164 | 228164 | changed |
| total cloud cover units | (0 - 1) | % | changed: values are 100 times larger |
| shortName | `sd`, `tcc` | `sd`, `tcc` | same |

Always identify parameters by **paramId**, and check the `units` key.

The old paramIds are **not** available in GRIB2. Asking for snow depth with `param=141` returns no data:

In [ ]:
try:
    ekd.from_source(
        "mars",
        {
            "class": "od",
            "stream": "oper",
            "type": "fc",
            "expver": "2250",
            "date": "20260901",
            "time": "00",
            "levtype": "sfc",
            "param": "141",           # old paramId: sd in m of water equivalent
            "timespan": "none",
            "step": "24",
            "grid": "0.25/0.25",
            "area": "75/-30/25/45",
        },
    ).to_fieldlist()
except Exception as e:
    print("Request failed:", e)

### Large-scale precipitation

`lsp` is accumulated from the start of the forecast, so the GRIB2 request also needs `timespan=fs` (see
practical 1). That is why it cannot go in the same request as `sd` and `tcc`, which have `timespan=none`.

#### GRIB1

In [ ]:
lsp_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "1",                # changed
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "142",               # changed: lsp in m
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
lsp_grib1.ls(keys=keys + ["metadata.stepRange", "metadata.maximum"])

#### GRIB2

In [ ]:
lsp_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "2250",             # changed (test data only)
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "3062",              # changed: lsp in kg m**-2
        "timespan": "fs",             # NEW: accumulated from the start of the forecast
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
lsp_grib2.ls(keys=keys + ["metadata.stepRange", "metadata.maximum"])

| | GRIB1 | GRIB2 | |
| --- | --- | --- | --- |
| `param` | 142 | 3062 | changed |
| units | m | kg m\*\*-2 (= mm) | changed: values are 1000 times larger |
| `timespan` | - | `fs` | new |
| shortName | `lsp` | `lsp` | same |

Note that the new paramId is **3062**, not 228142: the new paramIds do not follow a single pattern, so look
them up in the table below.

### Parameters provided in WMO units

| GRIB1 paramId (units) | GRIB2 paramId (units) | shortName |
| --- | --- | --- |
| 228 `tp` (m) | **228228** (kg m\*\*-2) | `tp` |
| 141 `sd` (m of water equivalent) | **228141** (kg m\*\*-2) | `sd` |
| 142 `lsp` (m) | **3062** (kg m\*\*-2) | `lsp` |
| 143 `cp` (m) | **228143** (kg m\*\*-2) | `cp` |
| 144 `sf` (m of water equivalent) | **228144** (kg m\*\*-2) | `sf` |
| 182 `e` (m of water equivalent) | **260259** (kg m\*\*-2) | `eva` |
| 205 `ro` (m) | **231002** (kg m\*\*-2) | `rowe` |
| 164 `tcc` (0 - 1) | **228164** (%) | `tcc` |
| 186 / 187 / 188 `lcc` / `mcc` / `hcc` (0 - 1) | **3073** / **3074** / **3075** (%) | `lcc` / `mcc` / `hcc` |
| 206 `tco3` (kg m\*\*-2) | **260132** (Dobson Units) | `tcioz` |

Accumulated parameters such as `tp`, `cp` and `sf` also need `timespan=fs` (see practical 1). The full list,
including albedos and runoff components, is in
[Migration to GRIB2 - changes to encoding of parameters](https://confluence.ecmwf.int/spaces/MTG2US/pages/554148197/Migration+to+GRIB2+-+changes+to+encoding+of+parameters#MigrationtoGRIB2changestoencodingofparameters-ParametersprovidedwithWMOunits).

## Wave parameters move to the atmospheric streams

Today the wave model output has its own MARS streams. In GRIB2 these are discontinued, and wave parameters
are stored alongside the atmospheric ones: `wave` becomes **`oper`**, `waef` becomes **`enfo`**.

### GRIB1

In [ ]:
wave_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "wave",             # changed
        "type": "fc",
        "expver": "1",                # changed
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "140229/140232",     # same: swh/mwp
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
wave_grib1.ls(keys=keys + ["metadata.maximum"])

### GRIB2

In [ ]:
wave_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",             # changed: wave -> oper
        "type": "fc",
        "expver": "2250",             # changed (test data only)
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "140229/140232",     # same: swh/mwp
        "timespan": "none",           # NEW: instantaneous fields
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
wave_grib2.ls(keys=keys + ["metadata.maximum", "metadata.discipline"])

| | GRIB1 | GRIB2 | |
| --- | --- | --- | --- |
| `stream` | `wave` | `oper` | changed |
| `param` | 140229/140232 | 140229/140232 | same |
| fields and values | `swh`, `mwp` | `swh`, `mwp` | same |

Only the stream changes. The paramIds start with 140 (wave), and in GRIB2 the wave parameters have their own
discipline (10, oceanographic products), so they remain easy to tell apart from the atmospheric ones.
`stream=wave` with `expver=2250` returns no data.

| class | GRIB1 stream | GRIB2 stream |
| --- | --- | --- |
| od | `wave` - wave model | `oper` |
| od | `waef` - wave ensemble forecast | `enfo` |
| od | `ewla` - ensemble wave long window data assimilation | `elda` |
| od | `enwh` - ensemble forecast wave hindcasts | `enfh` |
| od | `wehs` - wave ensemble forecast hindcast statistics | `efhs` |
| od | `weef` - sub-seasonal wave forecast | `eefo` |
| od | `weeh` - sub-seasonal wave forecast hindcasts | `eehs` |

## Your turn now

### Practical A
Modify the requests below to download low, medium and high cloud cover. (You can find the paramIds
[here](https://confluence.ecmwf.int/spaces/MTG2US/pages/554148197/Migration+to+GRIB2+-+changes+to+encoding+of+parameters#MigrationtoGRIB2changestoencodingofparameters-ParametersencodedbyECMWFasfractionalvaluesbetween0and1))

In [ ]:
cc_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "1",
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "",                  # Add paramIds here
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
cc_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "oper",
        "type": "fc",
        "expver": "2250",             # (test data only)
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "",                  # Add paramIds here
        "timespan": "",               # Add the timespan here
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

Compare the two. What changes, and what stays the same? By what factor do the values differ? (Hint: add
`metadata.units` and `metadata.maximum` to the keys in `ls`)

In [ ]:
cc_grib1.ls()

In [ ]:
cc_grib2.ls()

### Practical B
Modify the requests below to download significant wave height and mean wave period from **ensemble member 1**.
In GRIB1 the wave ensemble is in `stream=waef`. Which stream do you need for GRIB2?

In [ ]:
ens_wave_grib1 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "",                 # Add the stream here
        "type": "pf",
        "number": "1",
        "expver": "1",
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "",                  # Add paramIds here
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

In [ ]:
ens_wave_grib2 = ekd.from_source(
    "mars",
    {
        "class": "od",
        "stream": "",                 # Add the stream here
        "type": "pf",
        "number": "1",
        "expver": "2250",             # (test data only)
        "date": "20260901",
        "time": "00",
        "levtype": "sfc",
        "param": "",                  # Add paramIds here
        "timespan": "",               # Add the timespan here
        "step": "24",
        "grid": "0.25/0.25",
        "area": "75/-30/25/45",
    },
).to_fieldlist()

Compare the two. What changes, and what stays the same? 

In [ ]:
ens_wave_grib1.ls()

In [ ]:
ens_wave_grib2.ls()

Further reading:
- [Migration to GRIB2 - Test GRIB2 data in MARS](https://confluence.ecmwf.int/pages/viewpage.action?pageId=716086411)
- [Changes in the MARS language](https://confluence.ecmwf.int/pages/viewpage.action?pageId=621051006)
- [Migration to GRIB2 - changes to encoding of parameters](https://confluence.ecmwf.int/pages/viewpage.action?pageId=554148197)